In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import nfl_data_py as nfl
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score
from scipy import stats
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

df = pd.read_csv('../merged_data_with_college.csv')

In [2]:
df_qb = df[df['pos_group'] == 'QB']
for pct in [70, 75, 80, 85]:
    val = df_qb['w_av'].quantile(pct/100)
    print(f"{pct}th percentile: {val}")

70th percentile: 26.0
75th percentile: 40.0
80th percentile: 48.0
85th percentile: 61.0


In [3]:
print(df[(df['w_av'] >= 40) & (df['pos'] == 'QB')][['player_name', 'college', 'pos', 'w_av', 'pick']])

            player_name             college pos   w_av  pick
0             Tom Brady            Michigan  QB  184.0   199
1           Marc Bulger       West Virginia  QB   57.0   168
6       Chad Pennington            Marshall  QB   55.0    18
10           Drew Brees              Purdue  QB  167.0    32
16         Michael Vick       Virginia Tech  QB   92.0     1
18           David Carr          Fresno St.  QB   45.0     1
22        David Garrard       East Carolina  QB   62.0   108
25          Josh McCown     Sam Houston St.  QB   40.0    81
33        Carson Palmer                 USC  QB  107.0     1
39          Eli Manning         Mississippi  QB  121.0     1
45        Philip Rivers  North Carolina St.  QB  150.0     4
46   Ben Roethlisberger          Miami (OH)  QB  131.0    11
47          Matt Schaub            Virginia  QB   68.0    90
49       Jason Campbell              Auburn  QB   48.0    25
50     Ryan Fitzpatrick             Harvard  QB   78.0   250
53           Kyle Orton 

## we'll go with 75th percentile. all of these names are good QBs who got multi year deals at some point in their careers.

In [4]:
df_qb['is_hit'] = (df_qb['w_av'] >= 40).astype(int)
print(f"Hits: {df_qb['is_hit'].sum():.0f}, Busts: {(df_qb['is_hit'] == 0).sum()}")

Hits: 59, Busts: 212


In [5]:
features = ['ht', 'wt', 'forty', 'vertical', 'broad_jump']
df_qb_clean = df_qb.dropna(subset=features + ['is_hit'])
X = df_qb_clean[features]
y = df_qb_clean['is_hit']

print(f"Shape: {X.shape}")
print(y.value_counts())

Shape: (177, 5)
is_hit
0    138
1     39
Name: count, dtype: int64


In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size= 0.25,
                                                    random_state=42,
                                                    stratify=y)

## standardizing data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## running LogisiticRegression
model = LogisticRegression(random_state=42, class_weight='balanced')
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(classification_report(y_test,y_pred))

## running RandomForest
rf_model = RandomForestClassifier(random_state=42, class_weight='balanced')
rf_model.fit(X_train_scaled, y_train)
rf_pred = rf_model.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, rf_pred):.3f}")
print(classification_report(y_test,rf_pred))

## running GradientBoost
gb_model = GradientBoostingClassifier(random_state=42)
gb_model.fit(X_train_scaled, y_train)
gb_pred = gb_model.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, gb_pred):.3f}")
print(classification_report(y_test, gb_pred))

for name, mod in [('LogReg', LogisticRegression(random_state=42, class_weight='balanced')),
                   ('RF', RandomForestClassifier(random_state=42, class_weight='balanced')),
                   ('GB', GradientBoostingClassifier(random_state=42))]:
    scores = cross_val_score(mod, X_train_scaled, y_train, cv=5, scoring='accuracy')
    print(f"{name}: {scores.mean():.3f} (+/- {scores.std():.3f})")

Accuracy: 0.556
              precision    recall  f1-score   support

           0       0.78      0.60      0.68        35
           1       0.22      0.40      0.29        10

    accuracy                           0.56        45
   macro avg       0.50      0.50      0.48        45
weighted avg       0.65      0.56      0.59        45

Accuracy: 0.778
              precision    recall  f1-score   support

           0       0.80      0.94      0.87        35
           1       0.50      0.20      0.29        10

    accuracy                           0.78        45
   macro avg       0.65      0.57      0.58        45
weighted avg       0.74      0.78      0.74        45

Accuracy: 0.644
              precision    recall  f1-score   support

           0       0.76      0.80      0.78        35
           1       0.12      0.10      0.11        10

    accuracy                           0.64        45
   macro avg       0.44      0.45      0.44        45
weighted avg       0.62   

### Random forest performed really well. low hit recall, but that is expected since this is the combine-only feature model. most of the time physicals don't tell you much about success. 

in RF:
- bust precision is 0.80: when it predicts bust, it is correct 80% of the time
- hit precision is 0.50: when it predicts hit, it is correct 50% of the time
- bust recall is 0.94: so it found 94% of the actual busts
- hit recall is 0.20: so it found 20% of the actual hits

The accuracy of 77.8% sounds good but is mostly misleading because the model is just guessing bust most of the time.

For the report the most important number is hit recall, as that's how many successful players our model actual identified. A model that never predicts hit is useless for finding undervalued players, even if its accuracy is high.

In [7]:
college_features = ['career_g', 'career_cmp', 'career_att', 'career_cmp_pct',
                    'career_yds', 'career_td', 'career_int', 'career_rate',
                    'last_g', 'last_att', 'last_cmp_pct',
                    'last_yds', 'last_td', 'last_int', 'last_rate']

all_features = features + college_features
df_qb_college = df_qb.dropna(subset=all_features + ['is_hit'])
print(f"QBs with all features: {len(df_qb_college)}")
print(df_qb_college['is_hit'].value_counts())

QBs with all features: 166
is_hit
0    130
1     36
Name: count, dtype: int64


In [8]:
X = df_qb_college[all_features]
y = df_qb_college['is_hit']

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size= 0.25,
                                                    random_state=42,
                                                    stratify=y)

## standardizing data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## running LogisiticRegression
model = LogisticRegression(random_state=42, class_weight='balanced')
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(classification_report(y_test,y_pred))

## running RandomForest
rf_model = RandomForestClassifier(random_state=42, class_weight='balanced')
rf_model.fit(X_train_scaled, y_train)
rf_pred = rf_model.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, rf_pred):.3f}")
print(classification_report(y_test,rf_pred))

## running GradientBoost
gb_model = GradientBoostingClassifier(random_state=42)
gb_model.fit(X_train_scaled, y_train)
gb_pred = gb_model.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, gb_pred):.3f}")
print(classification_report(y_test, gb_pred))

for name, mod in [('LogReg', LogisticRegression(random_state=42, class_weight='balanced')),
                   ('RF', RandomForestClassifier(random_state=42, class_weight='balanced')),
                   ('GB', GradientBoostingClassifier(random_state=42))]:
    scores = cross_val_score(mod, X_train_scaled, y_train, cv=5, scoring='accuracy')
    print(f"{name}: {scores.mean():.3f} (+/- {scores.std():.3f})")

Accuracy: 0.595
              precision    recall  f1-score   support

           0       0.81      0.64      0.71        33
           1       0.25      0.44      0.32         9

    accuracy                           0.60        42
   macro avg       0.53      0.54      0.52        42
weighted avg       0.69      0.60      0.63        42

Accuracy: 0.810
              precision    recall  f1-score   support

           0       0.80      1.00      0.89        33
           1       1.00      0.11      0.20         9

    accuracy                           0.81        42
   macro avg       0.90      0.56      0.55        42
weighted avg       0.85      0.81      0.74        42

Accuracy: 0.810
              precision    recall  f1-score   support

           0       0.82      0.97      0.89        33
           1       0.67      0.22      0.33         9

    accuracy                           0.81        42
   macro avg       0.74      0.60      0.61        42
weighted avg       0.79   

#### random forest CV increased by 2.5%, accuracy by 3.2%
#### gradient boosting CV increased by 2.8%, accuracy by 16.6%
#### logistic regressing CV increased by 14%, accuracy by 3.9%

GB correctly predicts a hit 67% of the time (precision). The hit recall is low across all models which shows the predicting elite QBs is genuinely hard but that the college stats clearly added signal. Every single model improved on CV

The best model is GB, because RF has a 11% hit recall so it almost never predicts a hit. Whereas GB has 67% hit precision with a 22% recall. It predicts hit more often and is still right 2/3 of the time. When finding undervalued players we actually need to model to identify hits so a model like RF that almost never says hit is useless for that purpose

### Lets see if adjusting the prediction threshold for GB changes anything

In [9]:
## Adjust prediction threshold for GB
probs = gb_model.predict_proba(X_test_scaled)[:, 1]

for thresh in [0.50, 0.40, 0.30, 0.20]:
    pred = (probs >= thresh).astype(int)
    print(f"\nThreshold: {thresh}")
    print(classification_report(y_test, pred))


Threshold: 0.5
              precision    recall  f1-score   support

           0       0.82      0.97      0.89        33
           1       0.67      0.22      0.33         9

    accuracy                           0.81        42
   macro avg       0.74      0.60      0.61        42
weighted avg       0.79      0.81      0.77        42


Threshold: 0.4
              precision    recall  f1-score   support

           0       0.82      0.94      0.87        33
           1       0.50      0.22      0.31         9

    accuracy                           0.79        42
   macro avg       0.66      0.58      0.59        42
weighted avg       0.75      0.79      0.75        42


Threshold: 0.3
              precision    recall  f1-score   support

           0       0.81      0.91      0.86        33
           1       0.40      0.22      0.29         9

    accuracy                           0.76        42
   macro avg       0.61      0.57      0.57        42
weighted avg       0.72   

### The model is confident about the same QBs regardless of threshold. Lowering just adds more false positives without finding new hits. QB at 0.50 means high confidente and fewer picks. Which makes sense since it is a higher-stake pick so we would want more certainty
## Creating Draft Prediction Model

In [10]:
features = ['ht', 'wt', 'forty', 'vertical', 'broad_jump']
college_features = ['career_g', 'career_cmp', 'career_att', 'career_cmp_pct',
                    'career_yds', 'career_td', 'career_int', 'career_rate',
                    'last_g', 'last_att', 'last_cmp_pct',
                    'last_yds', 'last_td', 'last_int', 'last_rate']
all_features = features + college_features

In [11]:
df_qb_draft = df_qb.dropna(subset=all_features + ['pick'])
X = df_qb_draft[all_features]
y = df_qb_draft['pick']

print(f"Shape: {X.shape}")
print(f"Pick range: {y.min()} - {y.max()}")
print(f"Mean pick: {y.mean():.1f}, Median pick: {y.median():.1f}")

Shape: (166, 20)
Pick range: 1 - 253
Mean pick: 108.2, Median pick: 103.5


In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train.shape}, Test: {X_test.shape}\n")

for name, mod in [('Ridge', Ridge(random_state=42)),
                   ('Lasso', Lasso(random_state=42)),
                   ('RF', RandomForestRegressor(random_state=42)),
                   ('GB', GradientBoostingRegressor(random_state=42))]:
    mod.fit(X_train_scaled, y_train)
    pred = mod.predict(X_test_scaled)
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)
    print(f"{name}:")
    print(f"  MAE: {mae:.1f} picks off")
    print(f"  RMSE: {rmse:.1f}")
    print(f"  R²: {r2:.3f}")
    scores = cross_val_score(mod, X_train_scaled, y_train, cv=5, scoring='neg_mean_absolute_error')
    print(f"  CV MAE: {-scores.mean():.1f} (+/- {scores.std():.1f})\n")

Train: (124, 20), Test: (42, 20)

Ridge:
  MAE: 61.6 picks off
  RMSE: 76.1
  R²: 0.022
  CV MAE: 63.9 (+/- 6.1)

Lasso:
  MAE: 60.8 picks off
  RMSE: 75.2
  R²: 0.045
  CV MAE: 61.6 (+/- 6.8)

RF:
  MAE: 62.8 picks off
  RMSE: 76.0
  R²: 0.025
  CV MAE: 62.8 (+/- 8.6)

GB:
  MAE: 67.9 picks off
  RMSE: 82.4
  R²: -0.146
  CV MAE: 66.6 (+/- 8.5)



### MAE: tells us on average how many picks off is the prediction. MAE of 61 means our model is about 61 picks away from where they actually went. If we were to explain it to a GM we would say our model is off by about 2 rounds on average.

### RMSE: similar to MSE but penalizes big misses more heavily. if it's off by 5 picks for most players but 150 for one, RMSE will be much higher than MAE. When RMSE is higher than MAE it means some predicitons are way off even if most are reasonable.

### R-squared: tells us what % of the variance in draft position is explained by our model. Our model was near 0 which tells us that combine stats and college production barely explain why one QB goes 1st overall and another goes 200th. THIS IS EXPECTED!!!! OTHER FACTORS LIKE FILM, INTERVIEWS, ARM TALENT, ETC. WILL DRIVE UP A QBS DRAFT STOCK.

### CV MAE: same as MAE but averaged across 5 different train/test splits. This is the honest number. A single MAE could be lucky or unlucky depending on which players are in the test set. THIS IS THE NUMBER WE SHOULD REPORT.

## FOR THE REPORT: LEAD WITH CV MAE (5 FOLDS) SINCE IT IS THE MOST RELAIBLE AND EASIEST TO UNDERSTAND. USE R-SQUARED TO EXPLAIN HOW MUCH PREDICTIVE POWER THE MODEL HAS.

# Now I'm creating the FULL MODEL that uses the best hit/bust model and draft model to find undervalued QBs

In [13]:
## Find Undervalued QBs
## Use the best hit/bust model (GB) and best draft model (Lasso)

## Refit on all data (not just train)
X_all = df_qb_college[all_features]
y_hit = df_qb_college['is_hit']
y_pick = df_qb_college['pick']

scaler_all = StandardScaler()
X_all_scaled = scaler_all.fit_transform(X_all)

## Hit probability
gb_model = GradientBoostingClassifier(random_state=42)
gb_model.fit(X_all_scaled, y_hit)
hit_prob = gb_model.predict_proba(X_all_scaled)[:, 1]

## Predicted pick
lasso_model = Lasso(random_state=42)
lasso_model.fit(X_all_scaled, y_pick)
pred_pick = lasso_model.predict(X_all_scaled)

## Combine results
results = df_qb_college[['player_name', 'college', 'season', 'pick', 'w_av', 'is_hit']].copy()
results['hit_probability'] = hit_prob.round(3)
results['predicted_pick'] = pred_pick.round(1)
results['actual_pick'] = results['pick']
results['pick_difference'] = results['actual_pick'] - results['predicted_pick']

## Undervalued = high hit probability + drafted later than predicted
print("TOP QB PROSPECTS (highest hit probability): ")
print(results[results['predicted_pick'] > 0].sort_values('hit_probability', ascending=False).head(20)[
    ['player_name', 'college', 'season', 'actual_pick', 'predicted_pick', 'hit_probability', 'w_av']
].to_string(index=False))

TOP QB PROSPECTS (highest hit probability): 
      player_name         college  season  actual_pick  predicted_pick  hit_probability  w_av
   Deshaun Watson         Clemson    2017           12            72.1            0.977  58.0
       Geno Smith   West Virginia    2013           39            51.6            0.968  57.0
   Justin Herbert          Oregon    2020            6            31.8            0.953  74.0
   Jameis Winston     Florida St.    2015            1            94.2            0.951  63.0
       Jared Goff      California    2016            1            88.2            0.949 106.0
   Russell Wilson       Wisconsin    2012           75           112.6            0.946 140.0
       Joe Flacco        Delaware    2008           18            67.3            0.944  97.0
       Derek Carr      Fresno St.    2014           36            95.6            0.942  93.0
   Baker Mayfield        Oklahoma    2018            1            94.2            0.942  80.0
  Patrick Mahom

In [14]:
## Value score: high hit probability + drafted later than model expected
results['value_score'] = results['hit_probability'] * 100 + results['pick_difference']

print("\nMOST UNDERVALUED QBs (hit probability > 0.5 AND drafted later than predicted):")
undervalued = results[(results['hit_probability'] > 0.5) & (results['pick_difference'] > 0) & (results['predicted_pick'] > 0)]
print(undervalued.sort_values('value_score', ascending=False)[
    ['player_name', 'college', 'season', 'actual_pick', 'predicted_pick', 'hit_probability', 'w_av', 'value_score']
].head(15).to_string(index=False))


MOST UNDERVALUED QBs (hit probability > 0.5 AND drafted later than predicted):
 player_name         college  season  actual_pick  predicted_pick  hit_probability  w_av  value_score
Tyrod Taylor   Virginia Tech    2011          180           106.6            0.935  50.0        166.9
   Tom Brady        Michigan    2000          199           171.2            0.903 184.0        118.1
Dak Prescott Mississippi St.    2016          135           116.4            0.822 104.0        100.8
 Matt Schaub        Virginia    2004           90            88.8            0.882  68.0         89.4


## !!!!!!For the report:

We combined two models to identify undervalued draft picks. The first model (Gradient Boosting Classifier) predicts the probability that a QB will have a successful NFL career based on their combine measurable and college stats. The second model (Lasso regression) predicts where a QB should be drafted based on those same features. 

The hit/bust model tells us "how good will this player actually be?" and the draft prediction model tells us "how good do NFL teams think this player will be?" When a player has a high hit probability but a later predicted draft pick, it means the model sees something that NFL teams are undervaluing. This gap is where smart teams can find value.

We created a simple value sore that combines both sigals: hit probability plus pick difference. A positive pick difference means the player was drafted later than expected, so a high value score means the player is both likely to succeed AND was available later than he should have been. 

We only look at players with hit probability > 50% who were drafted later than the model predicted. This removes busts and overdrafted players, leaving only the true value picks.

Our model identified players like Tom Brady (pick 199, 90% hit probability), Dak Prescott (pick 135, 82% hit probability), and Tyrod Taylor (pick 180, 94% hit probability) as undervalued. A team who used this model would have flagged these players as high-value targets in later rounds and saved significant draft capital while still acquiring franchise-level talent.